# 07 - Final Predictions and Kaggle Submission

## Predicting Credit Card Fraud

### Objectives

- Rebuild the final feature engineering pipeline.
- Train optimized LightGBM and XGBoost models.
- Apply Random Oversampling.
- Generate fraud probabilities for the test dataset.
- Create Kaggle submission files.
- Validate submission structure.
- Compare public leaderboard results.

### Final configuration

- Feature set: Engineered Features.
- Models: LightGBM and XGBoost.
- Sampling strategy: Random Oversampling.
- Sampling ratio: 0.1.
- Training data: Full labeled training dataset.
- Evaluation metric: ROC-AUC.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import time

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (10, 5)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

In [2]:
possible_paths = [
    Path.cwd() / "data" / "raw",
    Path.cwd().parent / "data" / "raw"
]

DATA_DIR = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate data/raw directory"
    )

train = pd.read_csv(DATA_DIR / "train.csv")

test = pd.read_csv(DATA_DIR / "test.csv")

sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("TRAIN:", train.shape)
print("TEST:", test.shape)
print("SAMPLE SUBMISSION:", sample_submission.shape)

TRAIN: (699960, 26)
TEST: (299984, 25)
SAMPLE SUBMISSION: (299984, 2)


In [3]:
TARGET = "is_fraud"
ID_COL = "id"

# Check target and ID
assert TARGET in train.columns
assert ID_COL in train.columns

assert ID_COL in test.columns
assert TARGET not in test.columns

# Check train/test feature compatibility
train_features = set(train.columns) - {TARGET, ID_COL}

test_features = set(test.columns) - {ID_COL}

assert train_features == test_features

# Check target values
assert train[TARGET].notna().all()

assert set(train[TARGET].unique()) == {0.0, 1.0}

# Check sample submission
assert sample_submission.columns.tolist() == [
    "id",
    "is_fraud"
]

assert sample_submission["id"].equals(test["id"])

print("Dataset structure verified successfully.")

Dataset structure verified successfully.


In [4]:
def engineer_features(df):

    df = df.copy()

    # -----------------------------------
    # 1. Ratios and activity features
    # -----------------------------------

    df["amount_to_abs_balance_ratio"] = (
        df["amount_usd"] /
        (df["account_balance_usd"].abs() + 1)
    )

    df["cvv_retries_per_recent_txn"] = (
        df["cvv_retry_count"] /
        (df["txn_count_last_24h"] + 1)
    )

    # -----------------------------------
    # 2. Risk flag interactions
    # -----------------------------------

    df["vpn_ip_mismatch"] = (
        df["used_vpn"] &
        df["ip_country_mismatch"]
    )

    df["foreign_ip_mismatch"] = (
        df["is_foreign_transaction"] &
        df["ip_country_mismatch"]
    )

    df["foreign_billing_mismatch"] = (
        df["is_foreign_transaction"] &
        df["billing_shipping_mismatch"]
    )

    security_flags = [
        "is_foreign_transaction",
        "is_new_merchant",
        "used_vpn",
        "ip_country_mismatch",
        "billing_shipping_mismatch"
    ]

    df["security_risk_flags_count"] = (
        df[security_flags]
        .astype("int8")
        .sum(axis=1)
        .astype("int8")
    )

    # -----------------------------------
    # 3. Numerical interactions
    # -----------------------------------

    df["merchant_velocity_interaction"] = (
        df["merchant_risk_score"] *
        df["velocity_score"]
    )

    df["cvv_velocity_interaction"] = (
        df["cvv_retry_count"] *
        df["velocity_score"]
    )

    # -----------------------------------
    # 4. Logarithmic transformations
    # -----------------------------------

    log_features = [
        "amount_usd",
        "account_balance_usd",
        "cvv_retry_count",
        "prior_disputes"
    ]

    for col in log_features:

        df[f"log_{col}"] = (
            np.sign(df[col]) *
            np.log1p(np.abs(df[col]))
        )

    # -----------------------------------
    # 5. Cyclical encoding
    # -----------------------------------

    hour = df["time_of_day_hour"] % 24

    day = df["day_of_week"] % 7

    df["hour_sin"] = np.sin(
        2 * np.pi * hour / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * hour / 24
    )

    df["day_sin"] = np.sin(
        2 * np.pi * day / 7
    )

    df["day_cos"] = np.cos(
        2 * np.pi * day / 7
    )

    return df

In [5]:
# Preserve identifiers
train_ids = train[ID_COL].copy()

test_ids = test[ID_COL].copy()

# Separate predictors and target
X_train = train.drop(
    columns=[TARGET, ID_COL]
).copy()

y_train = train[TARGET].astype("int8").copy()

X_test = test.drop(
    columns=[ID_COL]
).copy()

# Preserve identical column ordering
X_test = X_test[X_train.columns]

# Apply feature engineering
X_train_fe = engineer_features(X_train)

X_test_fe = engineer_features(X_test)

print("Training features:", X_train_fe.shape)

print("Test features:", X_test_fe.shape)

print(
    "New features:",
    X_train_fe.shape[1] - X_train.shape[1]
)

Training features: (699960, 40)
Test features: (299984, 40)
New features: 16


In [6]:
assert X_train_fe.shape[1] == 40
assert X_test_fe.shape[1] == 40

assert X_train_fe.columns.equals(
    X_test_fe.columns
)

assert X_train_fe.isnull().sum().sum() == 0

assert X_test_fe.isnull().sum().sum() == 0

# Check numerical values
train_numeric = X_train_fe.select_dtypes(
    include="number"
)

test_numeric = X_test_fe.select_dtypes(
    include="number"
)

assert np.isfinite(
    train_numeric.to_numpy()
).all()

assert np.isfinite(
    test_numeric.to_numpy()
).all()

print("All features validated successfully.")

All features validated successfully.


In [7]:
def get_feature_types(df):

    bool_cols = [
        col for col in df.columns
        if pd.api.types.is_bool_dtype(df[col])
    ]

    cat_cols = [
        col for col in df.columns
        if (
            pd.api.types.is_object_dtype(df[col])
            or pd.api.types.is_string_dtype(df[col])
        )
    ]

    num_cols = [
        col for col in df.columns
        if (
            pd.api.types.is_numeric_dtype(df[col])
            and not pd.api.types.is_bool_dtype(df[col])
        )
    ]

    return num_cols, cat_cols, bool_cols


numeric_cols, categorical_cols, boolean_cols = (
    get_feature_types(X_train_fe)
)

print("Numerical:", len(numeric_cols))
print("Categorical:", len(categorical_cols))
print("Boolean:", len(boolean_cols))

Numerical: 26
Categorical: 5
Boolean: 9


In [8]:
def create_tree_preprocessor(df):

    num_cols, cat_cols, bool_cols = get_feature_types(df)

    try:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32
        )

    except TypeError:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32
        )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_cols),
            ("cat", encoder, cat_cols),
            ("bool", "passthrough", bool_cols)
        ],
        remainder="drop"
    )

    return preprocessor

In [9]:
final_model_configs = {

    "LightGBM": {

        "estimator": LGBMClassifier(

            objective="binary",

            n_estimators=400,

            learning_rate=0.03,

            num_leaves=31,

            max_depth=-1,

            min_child_samples=100,

            subsample=0.8,

            subsample_freq=1,

            colsample_bytree=0.7,

            reg_lambda=1,

            n_jobs=-1,

            random_state=42,

            verbosity=-1
        )

    },

    "XGBoost": {

        "estimator": XGBClassifier(

            objective="binary:logistic",

            eval_metric="auc",

            n_estimators=300,

            learning_rate=0.07,

            max_depth=3,

            min_child_weight=1,

            subsample=0.7,

            colsample_bytree=0.9,

            reg_lambda=0.5,

            gamma=0.1,

            tree_method="hist",

            n_jobs=-1,

            random_state=42,

            verbosity=0
        )

    }

}

print("Final models configured:")

for name in final_model_configs:

    print("-", name)

Final models configured:
- LightGBM
- XGBoost


In [10]:
def create_final_pipeline(estimator, X_reference):

    pipeline = Pipeline(
        steps=[

            (
                "sampler",
                RandomOverSampler(
                    sampling_strategy=0.1,
                    random_state=42
                )
            ),

            (
                "preprocessing",
                create_tree_preprocessor(X_reference)
            ),

            (
                "classifier",
                estimator
            )

        ]
    )

    return pipeline

In [11]:
final_models = {}

training_results = []

for model_name, config in final_model_configs.items():

    print("\n" + "=" * 60)
    print("Training final model:", model_name)
    print("=" * 60)

    model = create_final_pipeline(
        config["estimator"],
        X_train_fe
    )

    start_time = time.perf_counter()

    model.fit(
        X_train_fe,
        y_train
    )

    fit_time = time.perf_counter() - start_time

    # Number of rows after Random Oversampling
    resampled_rows = len(
        model.named_steps["sampler"].sample_indices_
    )

    final_models[model_name] = model

    training_results.append({

        "model": model_name,

        "original_training_rows": len(y_train),

        "resampled_training_rows": resampled_rows,

        "training_time_seconds": fit_time

    })

    print("Training completed.")

    print("Original rows:", len(y_train))

    print("Rows after sampling:", resampled_rows)

    print(f"Training time: {fit_time:.2f} seconds")


training_df = pd.DataFrame(
    training_results
).set_index("model")

display(training_df)


Training final model: LightGBM
Training completed.
Original rows: 699960
Rows after sampling: 757098
Training time: 9.30 seconds

Training final model: XGBoost
Training completed.
Original rows: 699960
Rows after sampling: 757098
Training time: 8.66 seconds


,original_training_rows,resampled_training_rows,training_time_seconds
model,,,
LightGBM,699960,757098,9.30413
XGBoost,699960,757098,8.66267


In [12]:
test_probabilities = {}

for model_name, model in final_models.items():

    print(f"\nGenerating predictions: {model_name}")

    y_proba = model.predict_proba(
        X_test_fe
    )[:, 1]

    # Validate probabilities
    assert len(y_proba) == len(test)

    assert np.isfinite(y_proba).all()

    assert np.all(y_proba >= 0)

    assert np.all(y_proba <= 1)

    test_probabilities[model_name] = y_proba

    print("Predictions:", len(y_proba))

    print(f"Minimum probability: {y_proba.min():.5f}")

    print(f"Maximum probability: {y_proba.max():.5f}")

    print(f"Mean probability: {y_proba.mean():.5f}")


Generating predictions: LightGBM
Predictions: 299984
Minimum probability: 0.00011
Maximum probability: 0.99235
Mean probability: 0.05109

Generating predictions: XGBoost
Predictions: 299984
Minimum probability: 0.00002
Maximum probability: 0.99929
Mean probability: 0.05264


In [13]:
prediction_summary = pd.DataFrame({

    "LightGBM": test_probabilities["LightGBM"],

    "XGBoost": test_probabilities["XGBoost"]

}).describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99
    ]
)

display(prediction_summary)

,LightGBM,XGBoost
count,"299,984.00000","299,984.00000"
mean,0.05109,0.05264
std,0.12754,0.12751
min,0.00011,0.00002
1%,0.00046,0.00025
5%,0.00092,0.00072
25%,0.00348,0.00365
50%,0.00971,0.01096
75%,0.03213,0.03479
95%,0.26308,0.26655


In [14]:
# LightGBM submission
submission_lightgbm = pd.DataFrame({

    "id": test_ids,

    "is_fraud": test_probabilities["LightGBM"]

})


# XGBoost submission
submission_xgboost = pd.DataFrame({

    "id": test_ids,

    "is_fraud": test_probabilities["XGBoost"]

})


# Equal-weight probability blend
blend_probabilities = (
    0.5 * test_probabilities["LightGBM"] +
    0.5 * test_probabilities["XGBoost"]
)

submission_blend = pd.DataFrame({

    "id": test_ids,

    "is_fraud": blend_probabilities

})


display(submission_lightgbm.head())

display(submission_xgboost.head())

display(submission_blend.head())

,id,is_fraud
0,699960,0.00161
1,699961,0.00144
2,699962,0.39264
3,699963,0.00919
4,699964,0.00814


,id,is_fraud
0,699960,0.00180
1,699961,0.00124
2,699962,0.41305
3,699963,0.00947
4,699964,0.01032


,id,is_fraud
0,699960,0.00171
1,699961,0.00134
2,699962,0.40285
3,699963,0.00933
4,699964,0.00923


In [15]:
submissions = {

    "LightGBM": submission_lightgbm,

    "XGBoost": submission_xgboost,

    "Blend": submission_blend

}

for name, submission in submissions.items():

    print(f"\nChecking {name}")

    # Exact columns
    assert submission.columns.tolist() == [
        "id",
        "is_fraud"
    ]

    # Correct number of rows
    assert len(submission) == len(test)

    # IDs match test
    assert submission["id"].equals(test_ids)

    # No missing values
    assert submission.isnull().sum().sum() == 0

    # Probabilities within [0, 1]
    assert submission["is_fraud"].between(
        0,
        1
    ).all()

    print("Rows:", len(submission))

    print("Columns:", submission.columns.tolist())

    print("IDs correctly matched.")

    print("Probabilities correctly formatted.")

print("\nAll submission files validated successfully.")


Checking LightGBM
Rows: 299984
Columns: ['id', 'is_fraud']
IDs correctly matched.
Probabilities correctly formatted.

Checking XGBoost
Rows: 299984
Columns: ['id', 'is_fraud']
IDs correctly matched.
Probabilities correctly formatted.

Checking Blend
Rows: 299984
Columns: ['id', 'is_fraud']
IDs correctly matched.
Probabilities correctly formatted.

All submission files validated successfully.


In [17]:
OUTPUT_DIR = Path("../outputs")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

submission_paths = {

    "LightGBM": OUTPUT_DIR / "submission_lightgbm.csv",

    "XGBoost": OUTPUT_DIR / "submission_xgboost.csv",

    "Blend": OUTPUT_DIR / "submission_blend.csv"

}

for name, submission in submissions.items():

    output_path = submission_paths[name]

    submission.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path.resolve()}")

Saved: C:\Users\afern\Desktop\Proyectos Personales\Kaggle - Credit Card Fraud\outputs\submission_lightgbm.csv
Saved: C:\Users\afern\Desktop\Proyectos Personales\Kaggle - Credit Card Fraud\outputs\submission_xgboost.csv
Saved: C:\Users\afern\Desktop\Proyectos Personales\Kaggle - Credit Card Fraud\outputs\submission_blend.csv


In [18]:
for name, path in submission_paths.items():

    saved_submission = pd.read_csv(path)

    print(f"\n{name}")

    print("Shape:", saved_submission.shape)

    print("Columns:", saved_submission.columns.tolist())

    display(saved_submission.head())

    assert saved_submission.shape == (
        len(test),
        2
    )

    assert saved_submission.columns.tolist() == [
        "id",
        "is_fraud"
    ]

print("\nAll exported submissions verified.")


LightGBM
Shape: (299984, 2)
Columns: ['id', 'is_fraud']


,id,is_fraud
0,699960,0.00161
1,699961,0.00144
2,699962,0.39264
3,699963,0.00919
4,699964,0.00814



XGBoost
Shape: (299984, 2)
Columns: ['id', 'is_fraud']


,id,is_fraud
0,699960,0.00180
1,699961,0.00124
2,699962,0.41305
3,699963,0.00947
4,699964,0.01032



Blend
Shape: (299984, 2)
Columns: ['id', 'is_fraud']


,id,is_fraud
0,699960,0.00171
1,699961,0.00134
2,699962,0.40285
3,699963,0.00933
4,699964,0.00923



All exported submissions verified.


# Final Project Conclusions

## 1. Final Models

- Final models: LightGBM and XGBoost.
- Feature set: 40 engineered features.
- Sampling strategy: Random Oversampling.
- Sampling ratio: 0.1.
- Training dataset: 699,960 observations.

## 2. Final Predictions

- Test observations: 299,984.
- LightGBM predictions generated successfully.
- XGBoost predictions generated successfully.
- Blended predictions generated successfully.
- All submission files passed structural validation.

## 3. Kaggle Results

| Model | Kaggle ROC-AUC |
|---|---:|
| LightGBM | 0.94651 |
| XGBoost | 0.94671 |
| Blend | 0.94689 |

The blended model achieved the best Kaggle score.

The ensemble combines LightGBM and XGBoost probabilities
using an equal-weight average.

## 4. Final Model Selection

The LightGBM/XGBoost blend was selected as the best
competition submission.

XGBoost achieved a very competitive individual score,
while LightGBM provided complementary predictions.

## 5. Project Limitations

- The dataset is synthetic.
- The public leaderboard may represent only a subset of test data.
- Probability calibration was not fully evaluated.
- The availability of `is_ai_generated_scam_attempt`
  requires further investigation before real-world deployment.
- Public leaderboard performance does not guarantee
  equivalent performance on real financial transactions.

## 6. Final Takeaways

- Class imbalance was a central challenge.
- Random Oversampling improved fraud detection at the
  default classification threshold.
- Feature Engineering improved Average Precision
  when combined with Random Oversampling.
- LightGBM and XGBoost were the strongest algorithms.
- Hyperparameter optimization improved Cross Validation results.
- Threshold optimization demonstrated the trade-off
  between Precision and Recall.
- Probability blending achieved the best Kaggle ROC-AUC.

## 7. Final Result

Best Kaggle ROC-AUC: 0.94689

Best submission: LightGBM + XGBoost Blend